# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es establecer un baseline con Multinomial Bayes Naive sobre dos representaciones de texto, Bag of Words y TF-IDF, con búsqueda de hiperparámetros.

NB es el modelo canónico para clasificación de texctos, es rápido de entrenar y transparente, ya que los pesos son log-probabilidades por palabra y clase.

Cualquier modelo mas sofisticado debe superar el F1-macro de NB, si no lo hace hay un bug.

## Importación e instalación de dependencias


In [1]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 89.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [3]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [4]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [5]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N3: Bayes Naive + TF-IDF con GridSearch


Se utiliza `GridSearchCV` para encontrar el mejor valor de `alpha`
(suavizado de Laplace) para Naive Bayes con representación TF-IDF.

`alpha` controla qué tan seguros está el modelo sobre palabras que
aparecen poco en el entrenamiento. Un alpha muy bajo puede hacer que
el modelo sea demasiado sensible a palabras raras; uno muy alto suaviza
demasiado y pierde discriminación.

GridSearch prueba 8 valores de alpha con validación cruzada de 5 folds
y elige el que maximiza el F1-macro promedio.

In [6]:
pipe = Pipeline([
        ("tfidf", make_tfidf()),
        ("nb",    MultinomialNB())])

param_grid_tfidf_v2 = {
    "nb__alpha": [0.05, 0.07, 0.1, 0.13, 0.15, 0.2, 0.3],
    "nb__fit_prior": [True, False]  # False = priors uniformes
}

gs_tfidf_v2 = GridSearchCV(
    pipe,
    param_grid_tfidf_v2,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

print("Buscando mejores hiperparámetros v2...")
gs_tfidf_v2.fit(X_train, y_train)

print(f"\nMejores hiperparámetros:: {gs_tfidf_v2.best_params_}")
print(f"Mejor F1-macro en CV: {gs_tfidf_v2.best_score_:.4f}")

Buscando mejores hiperparámetros v2...
Fitting 5 folds for each of 14 candidates, totalling 70 fits

Mejores hiperparámetros:: {'nb__alpha': 0.3, 'nb__fit_prior': False}
Mejor F1-macro en CV: 0.6470


In [7]:
best_tfidf_v2 = gs_tfidf_v2.best_estimator_
y_pred_tfidf_v2 = best_tfidf_v2.predict(X_val)

evaluate(
    "nb_tfidf_gridsearch_v2",
    y_val,
    y_pred_tfidf_v2,
    hyperparams={
        **gs_tfidf_v2.best_params_,
        "vectorizer": "TF-IDF",
        "cv_folds": 5
    }
)


=== nb_tfidf_gridsearch_v2 ===
Hiperparámetros: {'nb__alpha': 0.3, 'nb__fit_prior': False, 'vectorizer': 'TF-IDF', 'cv_folds': 5}

F1-macro:  0.6525
Precision: 0.6575
Recall:    0.6551
Accuracy:  0.6860

              precision    recall  f1-score   support

    negativa     0.7710    0.7203    0.7448      4080
      neutra     0.3781    0.5005    0.4308      2040
    positiva     0.8235    0.7444    0.7819      4080

    accuracy                         0.6860     10200
   macro avg     0.6575    0.6551    0.6525     10200
weighted avg     0.7134    0.6860    0.6969     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      2939     935       206
neutra         574    1021       445
positiva       299     744      3037


In [8]:
best_tfidf_v2.fit(
    np.concatenate([X_train, X_val]),
    np.concatenate([y_train, y_val])
)

save_model(best_tfidf_v2, "nb_tfidf_gridsearch")
make_submission(
    best_tfidf_v2,
    pd.DataFrame({"id": test["id"], "text": X_test}),
    "submission_nb_tfidf_gridsearch.csv"
)

Modelo guardado → models/nb_tfidf_gridsearch.joblib
Predicción guardada → submissions/submission_nb_tfidf_gridsearch.csv  (8500 predicciones)
Distribución: clase 0: 37.5%, clase 1: 26.0%, clase 2: 36.5%


PosixPath('submissions/submission_nb_tfidf_gridsearch.csv')

El GridSearch sobre TF-IDF encontró que la mejor combinación es
`alpha=0.3` con `fit_prior=False`, alcanzando un F1-macro de 0.6525
en el set de validación.

El parámetro más influyente resultó ser `fit_prior=False` (priors
uniformes), que fuerza al modelo a tratar las tres clases con igual
probabilidad a priori. Esto benefició especialmente a la clase neutra, ya que el modelo dejó de sesgar sus predicciones hacia las clases mayoritarias.

El alpha=0.3 (menor al default de 1.0) indica que el dataset es lo
suficientemente grande como para que el modelo confíe más en las
frecuencias observadas y necesite menos suavizado.